# FGAT - Notebook 3: Train Model

**Mục tiêu:** Train FGAT model và evaluate trên test set

**Input (output của Notebook 1 + 2):**
- `item_embs.npy`, `user_embs.npy`, `outfit_embs.npy`
- `item_item_adj.npz`, `outfit_item_adj.npz`, `user_outfit_adj.npz`
- `train_uo_split.txt`, `val_uo_split.txt`, `test_uo.txt`
- `train_fltb.txt`, `val_fltb.txt`, `test_fltb.txt`
- `item_data.txt`, `outfit_data.txt`, `user_data.txt`
- `item_id_order.npy`, `outfit_id_order.npy`, `user_id_order.npy`

**Output:** `best_model.pt`, training curves

In [43]:
# ============================================================
# CELL 1: Install torch-scatter khớp với PyTorch + CUDA trên Kaggle
# ============================================================
import subprocess, sys

# Detect PyTorch version và CUDA version đang chạy
result = subprocess.run(
    [sys.executable, '-c',
     'import torch; print(torch.__version__); print(torch.version.cuda)'],
    capture_output=True, text=True
)
torch_ver, cuda_ver = result.stdout.strip().split('\n')
print(f'PyTorch : {torch_ver}')
print(f'CUDA    : {cuda_ver}')

# Build whl URL đúng format pyg.org
# Ví dụ: torch==2.5.1, cuda==12.1  →  torch-2.5.0+cu121
major_minor = '.'.join(torch_ver.split('.')[:2])          # '2.5'
torch_tag   = f"{major_minor}.0"                          # '2.5.0'
cuda_tag    = 'cu' + cuda_ver.replace('.', '')[:3]        # 'cu121'
whl_url     = f'https://data.pyg.org/whl/torch-{torch_tag}+{cuda_tag}.html'
print(f'WHL URL : {whl_url}')

# Gỡ version cũ (nếu có) rồi cài lại đúng version
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch-scatter'], check=False)
ret = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'torch-scatter', '-f', whl_url],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print('WHL install failed, trying pip default...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-scatter'], check=True)
else:
    print('torch-scatter installed OK')

# Cài thêm các thư viện còn lại
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric', 'seaborn', 'scikit-learn'], check=True)

# Verify import
try:
    from torch_scatter import scatter_add
    print('torch_scatter import: OK')
except Exception as e:
    print(f'torch_scatter import FAILED: {e}')
    print('Fallback: will use manual scatter in model code')


PyTorch : 2.10.0+cu128
CUDA    : 12.8
WHL URL : https://data.pyg.org/whl/torch-2.10.0+cu128.html
Found existing installation: torch_scatter 2.1.2+pt210cu128
Uninstalling torch_scatter-2.1.2+pt210cu128:
  Successfully uninstalled torch_scatter-2.1.2+pt210cu128
torch-scatter installed OK
torch_scatter import: OK


In [44]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
KAGGLE_DATA = Path('/kaggle/input/datasets/kiettruonglifeez/recsys-fgat')
LOCAL_DATA  = REPO_ROOT / 'hfgat_rewrite_validate' / 'Dataset'

if KAGGLE_DATA.exists():
    DATA_ROOT   = KAGGLE_DATA
    OUTPUT_ROOT = Path('/kaggle/working')
    IMAGE_ROOT  = OUTPUT_ROOT / 'images'
elif LOCAL_DATA.exists():
    DATA_ROOT   = LOCAL_DATA
    OUTPUT_ROOT = REPO_ROOT / 'output_fgat_v1'
    IMAGE_ROOT  = LOCAL_DATA / 'fashion_item_images'
else:
    raise FileNotFoundError(
        f'Dataset not found. Expected {LOCAL_DATA} (local) or {KAGGLE_DATA} (Kaggle).'
    )

DATA_DIR  = str(DATA_ROOT) + '/'
OUT_DIR   = str(OUTPUT_ROOT) + '/'
IMAGE_DIR = str(IMAGE_ROOT) + '/'
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

FEAT_DIR = OUT_DIR
_kaggle_feat = Path('/kaggle/input/notebooks/kiettruonglifeez/fgat-session-2-text-graph')
if _kaggle_feat.exists():
    FEAT_DIR = str(_kaggle_feat) + '/'

EMBED_DIM   = 64
NUM_HEADS   = 4
DROPOUT     = 0.2
LR          = 0.001
WEIGHT_DECAY= 1e-5
BATCH_SIZE  = 512
NUM_EPOCHS  = 40
LAMBDA_COMP = 0.5
PATIENCE    = 10
MAX_ITEMS   = 10
K           = 10

SAVE_PATH = os.path.join(OUT_DIR, 'best_model.pt')
os.makedirs(OUT_DIR, exist_ok=True)
print('Config OK')
print(f'  FEAT_DIR : {FEAT_DIR}')
print(f'  DATA_DIR : {DATA_DIR}')
print(f'  OUT_DIR  : {OUT_DIR}')


Config OK


In [45]:
# ============================================================
# CELL 3: Imports + scatter_add fallback
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time
import random
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm

# scatter_add: dùng torch_scatter nếu có, fallback sang torch thuần
try:
    from torch_scatter import scatter_add
    print('Using torch_scatter.scatter_add')
except Exception:
    print('torch_scatter unavailable → using torch fallback')
    def scatter_add(src, index, dim=0, dim_size=None, out=None):
        """Pure-PyTorch fallback cho scatter_add."""
        if dim_size is None:
            dim_size = int(index.max().item()) + 1
        shape = list(src.shape)
        shape[dim] = dim_size
        result = src.new_zeros(shape)
        # index phải broadcast với src theo chiều dim
        idx = index
        for _ in range(src.dim() - 1):
            idx = idx.unsqueeze(-1)
        idx = idx.expand_as(src)
        result.scatter_add_(dim, idx, src)
        return result

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


Using torch_scatter.scatter_add
Device: cuda


In [46]:
# ============================================================
# CELL 4: Load embeddings & edge matrices
# ============================================================
def load_npz_edges(path):
    mat  = sp.load_npz(path)
    coo  = mat.tocoo()
    edge_index  = torch.tensor(np.vstack([coo.row, coo.col]), dtype=torch.long).to(device)
    edge_weight = torch.tensor(coo.data, dtype=torch.float32).to(device)
    return edge_index, edge_weight

def build_normalized_sparse(edge_index, edge_weight, n_row, n_col, device):
    """Tạo row-normalized sparse matrix."""
    row, col = edge_index
    # row normalize
    deg = torch.zeros(n_row, device=device)
    deg.scatter_add_(0, row, edge_weight)
    deg_inv = 1.0 / deg.clamp(min=1e-8)
    norm_w = edge_weight * deg_inv[row]
    return torch.sparse_coo_tensor(
        torch.stack([row, col]), norm_w,
        (n_row, n_col), device=device
    ).coalesce()

# Embeddings (learnable - chuyển sang nn.Parameter trong model)
item_embs_np   = np.load(os.path.join(FEAT_DIR, 'item_embs.npy'))
user_embs_np   = np.load(os.path.join(FEAT_DIR, 'user_embs.npy'))
outfit_embs_np = np.load(os.path.join(FEAT_DIR, 'outfit_embs.npy'))

item_embs   = F.normalize(torch.tensor(item_embs_np,   dtype=torch.float32), dim=-1).to(device)
user_embs   = F.normalize(torch.tensor(user_embs_np,   dtype=torch.float32), dim=-1).to(device)
outfit_embs = F.normalize(torch.tensor(outfit_embs_np, dtype=torch.float32), dim=-1).to(device)

# Edge matrices
item_item_index,   item_item_weight   = load_npz_edges(os.path.join(FEAT_DIR, 'item_item_adj.npz'))
outfit_item_index, outfit_item_weight = load_npz_edges(os.path.join(FEAT_DIR, 'outfit_item_adj.npz'))
user_outfit_index, user_outfit_weight = load_npz_edges(os.path.join(FEAT_DIR, 'user_outfit_adj.npz'))

# Build sparse matrices
n_items   = item_embs.shape[0]
n_outfits = outfit_embs.shape[0]
n_users   = user_embs.shape[0]

item_adj_sparse        = build_normalized_sparse(item_item_index,   item_item_weight,   n_items,   n_items,   device)
outfit_item_adj_sparse = build_normalized_sparse(outfit_item_index, outfit_item_weight, n_outfits, n_items,   device)
user_outfit_adj_sparse = build_normalized_sparse(user_outfit_index, user_outfit_weight, n_users,   n_outfits, device)

# ID mappings
item_id_order   = np.load(os.path.join(FEAT_DIR, 'item_id_order.npy'),   allow_pickle=True).tolist()
outfit_id_order = np.load(os.path.join(FEAT_DIR, 'outfit_id_order.npy'), allow_pickle=True).tolist()
user_id_order   = np.load(os.path.join(FEAT_DIR, 'user_id_order.npy'),   allow_pickle=True).tolist()

item2id   = {str(iid): i for i, iid in enumerate(item_id_order)}
outfit2id = {str(oid): i for i, oid in enumerate(outfit_id_order)}
user2id   = {str(uid): i for i, uid in enumerate(user_id_order)}

# Metadata
item_data   = pd.read_csv(os.path.join(DATA_DIR, 'item_data.txt'),   header=None, names=['item_id','category','image_url','title'])
outfit_data = pd.read_csv(os.path.join(DATA_DIR, 'outfit_data.txt'), header=None, names=['outfit_id','items'])
user_data   = pd.read_csv(os.path.join(DATA_DIR, 'user_data.txt'),   header=None, names=['user_id','outfit_id'])

print(f'item_embs  : {item_embs.shape}')
print(f'user_embs  : {user_embs.shape}')
print(f'outfit_embs: {outfit_embs.shape}')
print(f'item_item  edges: {item_item_index.shape[1]}')
print(f'outfit_item edges: {outfit_item_index.shape[1]}')
print(f'user_outfit edges: {user_outfit_index.shape[1]}')

item_embs  : torch.Size([19175, 64])
user_embs  : torch.Size([277469, 64])
outfit_embs: torch.Size([9373, 64])
item_item  edges: 103652
outfit_item edges: 36401
user_outfit edges: 679028


In [47]:
# ============================================================
# CELL 5: Dataset classes
# ============================================================
class OutfitRecommendationDataset(Dataset):
    def __init__(self, train_file, outfit_data, user2id, outfit2id, item2id, max_items=10):
        self.pairs    = []
        self.max_items = max_items
        outfit_items  = outfit_data.set_index(outfit_data['outfit_id'].astype(str))['items']

        with open(train_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 2:
                    continue
                uid = str(parts[0])
                for oid in parts[1:]:
                    oid = str(oid)
                    u_idx = user2id.get(uid, -1)
                    o_idx = outfit2id.get(oid, -1)
                    if u_idx == -1 or o_idx == -1:
                        continue
                    item_idxs = []
                    if oid in outfit_items.index:
                        for it in str(outfit_items[oid]).split(';'):
                            item_idxs.append(item2id.get(it.strip(), -1))
                    item_idxs = (item_idxs + [-1]*max_items)[:max_items]
                    self.pairs.append((u_idx, o_idx, item_idxs))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, o, items = self.pairs[idx]
        return {
            'user_idx':    torch.tensor(u,     dtype=torch.long),
            'outfit_idx':  torch.tensor(o,     dtype=torch.long),
            'item_indices':torch.tensor(items, dtype=torch.long)
        }


class CompatibilityDataset(Dataset):
    def __init__(self, file_path, item2id, max_items=10):
        self.samples  = []
        self.max_items = max_items

        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().split(';')
                if len(parts) < 5:
                    continue
                pos_items = [item2id.get(str(i), -1) for i in parts[3].split(',')]
                neg_items = [item2id.get(str(i), -1) for i in parts[4].split(',')]
                if all(x == -1 for x in pos_items):
                    continue
                pos_items = (pos_items + [-1]*max_items)[:max_items]
                neg_items = (neg_items + [-1]*max_items)[:max_items]
                self.samples.append((pos_items, neg_items))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pos, neg = self.samples[idx]
        return {
            'pos_item_indices': torch.tensor(pos, dtype=torch.long),
            'neg_item_indices': torch.tensor(neg, dtype=torch.long)
        }


print('Dataset classes defined')

Dataset classes defined


In [59]:
# ============================================================
# CELL 6: Model classes
# ============================================================
class SparseGATLayer(nn.Module):
    """Dùng sparse matrix multiply thay vì attention thủ công — không tạo edge tensor lớn."""
    def __init__(self, in_dim, out_dim, dropout=0.2):
        super().__init__()
        self.W   = nn.Linear(in_dim, out_dim, bias=False)
        self.bn  = nn.LayerNorm(out_dim)
        self.act = nn.LeakyReLU(0.2)
        self.drop = nn.Dropout(dropout)

    def forward(self, h, adj_sparse):
        # adj_sparse: torch.sparse_coo_tensor đã normalize
        Wh = self.W(h)
        agg = torch.sparse.mm(adj_sparse, Wh)  # ✅ không tạo [E, H, D] tensor
        return self.bn(self.act(agg))


class CompatibilityScorer(nn.Module):
    def __init__(self, dim, num_views=6, hidden_dim=256):
        super().__init__()
        self.W5 = nn.Linear(dim, hidden_dim)
        self.bn5= nn.LayerNorm(hidden_dim)
        self.W4 = nn.Linear(hidden_dim, num_views)
        self.W7 = nn.Linear(dim, hidden_dim)
        self.bn7= nn.LayerNorm(hidden_dim)
        self.W6 = nn.Linear(hidden_dim, num_views)
        self.leaky  = nn.LeakyReLU(0.2)

    def forward(self, outfit_embs):  # [B, max_items, dim]
        A = F.softmax(self.W4(self.leaky(self.bn5(self.W5(outfit_embs)))).transpose(1,2), dim=-1)
        C = torch.tanh(self.W6(self.leaky(self.bn7(self.W7(outfit_embs)))).transpose(1,2))
        return (A*C).sum(-1).sum(-1)


class H_HFGAT(nn.Module):
    def __init__(self, dim=64, dropout=0.2):
        super().__init__()
        self.item_gat             = SparseGATLayer(dim, dim, dropout)
        self.item_proj            = nn.Linear(dim, dim)
        self.user_proj            = nn.Linear(dim, dim)
        self.compatibility_scorer = CompatibilityScorer(dim)
        self.dropout              = nn.Dropout(dropout)
        self.act                  = nn.LeakyReLU(0.2)

    def forward(self, item_embs,   item_adj_sparse,
                      outfit_embs, outfit_item_adj_sparse,
                      user_embs,   user_outfit_adj_sparse):   # ← 3 sparse matrix, không phải 6 args

        item_upd   = F.normalize(self.dropout(self.item_gat(item_embs, item_adj_sparse)), p=2, dim=-1)
        outfit_upd = F.normalize(self.dropout(self.act(self.item_proj(
                         torch.sparse.mm(outfit_item_adj_sparse, item_upd)))), p=2, dim=-1)
        user_upd   = F.normalize(self.dropout(self.act(self.user_proj(
                         torch.sparse.mm(user_outfit_adj_sparse, outfit_upd) + user_embs))), p=2, dim=-1)

        return item_upd, outfit_upd, user_upd

    def score_recommendation(self, user_emb, outfit_emb):
        return torch.sum(user_emb * outfit_emb, dim=-1)

print('Model classes defined')

Model classes defined


In [60]:
# ============================================================
# CELL 7: Loss & Evaluation functions
# ============================================================
def bpr_loss(pos_score, neg_score):
    return -torch.mean(F.logsigmoid(pos_score - neg_score))


@torch.no_grad()
def evaluate_rec(model, loader,
                 item_embs,   item_adj_sparse,
                 outfit_embs, outfit_item_adj_sparse,
                 user_embs,   user_outfit_adj_sparse,
                 device, k=10):
    model.eval()
    item_upd, outfit_upd, user_upd = model(
        item_embs,   item_adj_sparse,
        outfit_embs, outfit_item_adj_sparse,
        user_embs,   user_outfit_adj_sparse
    )
    all_hr, all_prec, all_rec, all_ndcg, all_auc = [], [], [], [], []
    for batch in loader:
        u_idx = batch['user_idx'].to(device)
        o_idx = batch['outfit_idx'].to(device)
        for u in torch.unique(u_idx):
            mask    = u_idx == u
            outfits = o_idx[mask]
            pos_sc  = model.score_recommendation(user_upd[u].unsqueeze(0).expand(len(outfits), -1), outfit_upd[outfits])
            labels  = torch.ones(len(outfits), device=device)
            num_neg = 50
            neg_idx = torch.randint(outfit_upd.size(0), (num_neg,), device=device)
            neg_sc  = model.score_recommendation(user_upd[u].unsqueeze(0).expand(num_neg, -1), outfit_upd[neg_idx])
            sc_all  = torch.cat([pos_sc, neg_sc])
            lb_all  = torch.cat([labels, torch.zeros(num_neg, device=device)])
            _, topk = torch.topk(sc_all, k)
            hits    = lb_all[topk]
            hr      = (hits.sum() > 0).float().item()
            prec    = hits.sum().item() / k
            rec     = hits.sum().item() / labels.sum().item() if labels.sum() > 0 else 0.0
            dcg     = (hits / torch.log2(torch.arange(2, 2+k, device=device).float())).sum().item()
            idcg_k  = min(int(labels.sum().item()), k)
            idcg    = (1.0 / torch.log2(torch.arange(2, 2+idcg_k, device=device).float())).sum().item() if idcg_k > 0 else 0.0
            ndcg    = dcg / idcg if idcg > 0 else 0.0
            try:
                auc = roc_auc_score(lb_all.cpu().numpy(), sc_all.detach().cpu().numpy())
            except:
                auc = float('nan')
            all_hr.append(hr); all_prec.append(prec)
            all_rec.append(rec); all_ndcg.append(ndcg); all_auc.append(auc)
    return {
        'HR@K':        np.nanmean(all_hr),
        'Precision@K': np.nanmean(all_prec),
        'Recall@K':    np.nanmean(all_rec),
        'NDCG@K':      np.nanmean(all_ndcg),
        'AUC':         np.nanmean(all_auc),
    }


@torch.no_grad()
def evaluate_compat(model, loader, item_embs, device):
    model.eval()
    accs = []
    for batch in loader:
        pos_idx  = batch['pos_item_indices'].to(device)
        neg_idx  = batch['neg_item_indices'].to(device)
        pos_embs = item_embs[pos_idx]
        neg_embs = item_embs[neg_idx]
        pos_sc   = model.compatibility_scorer(pos_embs)   # ← sửa compat_scorer → compatibility_scorer
        neg_sc   = model.compatibility_scorer(neg_embs)
        accs.append((pos_sc > neg_sc).float().mean().item())
    return np.mean(accs) if accs else 0.0


print('Loss & eval functions defined')

Loss & eval functions defined


In [61]:
# ============================================================
# CELL: Split train_uo.txt → train / val / test (80/10/10)
# ============================================================
import random
import numpy as np

TRAIN_UO_FILE = os.path.join(DATA_DIR, 'train_uo.txt')

with open(TRAIN_UO_FILE, 'r') as f:
    uo_lines = f.readlines()

np.random.seed(42)
np.random.shuffle(uo_lines)

n         = len(uo_lines)
train_end = int(0.8 * n)
val_end   = int(0.9 * n)

TRAIN_UO_SPLIT = os.path.join(OUT_DIR, 'train_uo_split.txt')
VAL_UO_SPLIT   = os.path.join(OUT_DIR, 'val_uo_split.txt')
TEST_UO_SPLIT  = os.path.join(OUT_DIR, 'test_uo_split.txt')

with open(TRAIN_UO_SPLIT, 'w') as f:
    f.writelines(uo_lines[:train_end])
with open(VAL_UO_SPLIT, 'w') as f:
    f.writelines(uo_lines[train_end:val_end])
with open(TEST_UO_SPLIT, 'w') as f:
    f.writelines(uo_lines[val_end:])

print(f'train: {train_end} | val: {val_end-train_end} | test: {n-val_end}')


train: 221975 | val: 27747 | test: 27747


In [62]:
# ============================================================
# CELL: Generate FLTB files (train / val / test)
# Format: outfit_id;outfit_len;mask_pos;pos_items;neg1;neg2;neg3
# ============================================================
import pandas as pd
from sklearn.model_selection import train_test_split

# Load outfit_data (cần cho write_fltb)
outfit_data_fltb = pd.read_csv(
    os.path.join(DATA_DIR, 'outfit_data.txt'),
    header=None, names=['outfit_id', 'items'])
outfit_data_fltb['outfit_id'] = outfit_data_fltb['outfit_id'].astype(str)

all_outfit_items = outfit_data_fltb.copy()
all_outfit_items['items_list'] = all_outfit_items['items'].apply(
    lambda x: [it.strip() for it in str(x).split(';')])
all_outfit_items['length'] = all_outfit_items['items_list'].apply(len)
all_outfit_items = all_outfit_items[all_outfit_items['length'] >= 2].reset_index(drop=True)

all_item_pool = list(set(it for its in all_outfit_items['items_list'] for it in its))

train_df, temp_df = train_test_split(all_outfit_items, test_size=0.2, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)

def write_fltb(df, out_path):
    written = 0
    with open(out_path, 'w') as f:
        for _, row in df.iterrows():
            items    = [int(it) for it in row['items_list']]
            mask_pos = random.randint(0, len(items)-1)
            pos_str  = ','.join(map(str, items))
            neg_parts = []
            for _ in range(3):
                neg = items.copy()
                candidates = [it for it in all_item_pool if it not in map(str, items)]
                if not candidates:
                    break
                neg[mask_pos] = int(random.choice(candidates))
                neg_parts.append(','.join(map(str, neg)))
            if len(neg_parts) < 3:
                continue
            line = f"{row['outfit_id']};{len(items)};{mask_pos};{pos_str};{';'.join(neg_parts)}\n"
            f.write(line)
            written += 1
    return written

TRAIN_FLTB = os.path.join(OUT_DIR, 'train_fltb.txt')
VAL_FLTB   = os.path.join(OUT_DIR, 'val_fltb.txt')
TEST_FLTB  = os.path.join(OUT_DIR, 'test_fltb.txt')

n_train = write_fltb(train_df, TRAIN_FLTB)
n_val   = write_fltb(val_df,   VAL_FLTB)
n_test  = write_fltb(test_df,  TEST_FLTB)
print(f'FLTB - train: {n_train} | val: {n_val} | test: {n_test}')


FLTB - train: 7498 | val: 937 | test: 938


In [65]:
# ============================================================
# CELL 8: Create DataLoaders
# ============================================================
train_dataset = OutfitRecommendationDataset(
    TRAIN_UO_SPLIT,
    outfit_data, user2id, outfit2id, item2id, MAX_ITEMS)

val_dataset = OutfitRecommendationDataset(
    VAL_UO_SPLIT,
    outfit_data, user2id, outfit2id, item2id, MAX_ITEMS)

test_dataset = OutfitRecommendationDataset(
    TEST_UO_SPLIT,
    outfit_data, user2id, outfit2id, item2id, MAX_ITEMS)

compat_train_dataset = CompatibilityDataset(TRAIN_FLTB, item2id, MAX_ITEMS)
compat_val_dataset   = CompatibilityDataset(VAL_FLTB,   item2id, MAX_ITEMS)
compat_test_dataset  = CompatibilityDataset(TEST_FLTB,  item2id, MAX_ITEMS)

train_loader       = DataLoader(train_dataset,       batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader         = DataLoader(val_dataset,         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader        = DataLoader(test_dataset,        batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
compat_train_loader= DataLoader(compat_train_dataset,batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
compat_val_loader  = DataLoader(compat_val_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
compat_test_loader = DataLoader(compat_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)}')
print(f'compat train: {len(compat_train_dataset)} | val: {len(compat_val_dataset)} | test: {len(compat_test_dataset)}')

train: 543619 | val: 67606 | test: 67803
compat train: 7498 | val: 937 | test: 938


In [66]:
# ============================================================
# CELL 9: Init model, optimizer, scheduler
# ============================================================
model = H_HFGAT(dim=EMBED_DIM, dropout=DROPOUT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

Model parameters: 49,932
H_HFGAT(
  (item_gat): SparseGATLayer(
    (W): Linear(in_features=64, out_features=64, bias=False)
    (bn): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (act): LeakyReLU(negative_slope=0.2)
    (drop): Dropout(p=0.2, inplace=False)
  )
  (item_proj): Linear(in_features=64, out_features=64, bias=True)
  (user_proj): Linear(in_features=64, out_features=64, bias=True)
  (compatibility_scorer): CompatibilityScorer(
    (W5): Linear(in_features=64, out_features=256, bias=True)
    (bn5): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (W4): Linear(in_features=256, out_features=6, bias=True)
    (W7): Linear(in_features=64, out_features=256, bias=True)
    (bn7): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (W6): Linear(in_features=256, out_features=6, bias=True)
    (leaky): LeakyReLU(negative_slope=0.2)
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (act): LeakyReLU(negative_slope=0.2)
)


In [70]:
# ============================================================
# CELL 10: Training loop
# ============================================================
history = {k: [] for k in [
    'train_loss', 'val_loss', 'train_rec_loss', 'val_rec_loss',
    'train_comp_loss', 'val_comp_loss',
    'train_hr','val_hr','train_ndcg','val_ndcg',
    'train_prec','val_prec','train_rec','val_rec',
    'train_auc','val_auc','train_acc','val_acc'
]}

best_val_hr    = 0.0
early_stop_cnt = 0

print(f'Starting training for {NUM_EPOCHS} epochs...')

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    model.train()
    total_rec_loss = total_comp_loss = 0.0
    n_batches = 0
    compat_iter = iter(compat_train_loader)

    for batch in train_loader:
        optimizer.zero_grad()

        # Forward mỗi batch — graph tạo và giải phóng ngay
        item_upd, outfit_upd, user_upd = model(
            item_embs, item_adj_sparse,
            outfit_embs, outfit_item_adj_sparse,
            user_embs, user_outfit_adj_sparse
        )

        u_idx   = batch['user_idx'].to(device)
        o_idx   = batch['outfit_idx'].to(device)
        neg_idx = torch.randint(outfit_upd.size(0), o_idx.shape, device=device)

        loss_rec = bpr_loss(
            model.score_recommendation(user_upd[u_idx], outfit_upd[o_idx]),
            model.score_recommendation(user_upd[u_idx], outfit_upd[neg_idx])
        )

        try:
            cb = next(compat_iter)
        except StopIteration:
            compat_iter = iter(compat_train_loader)
            cb = next(compat_iter)

        pos_embs  = item_upd[cb['pos_item_indices'].to(device)]
        neg_embs  = item_upd[cb['neg_item_indices'].to(device)]
        loss_comp = bpr_loss(
            model.compatibility_scorer(pos_embs),
            model.compatibility_scorer(neg_embs)
        )

        loss = loss_rec + LAMBDA_COMP * loss_comp
        loss.backward()  # ✅ không retain_graph
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_rec_loss  += loss_rec.item()
        total_comp_loss += loss_comp.item()
        n_batches += 1

    torch.cuda.empty_cache()
    # Validation loss
    model.eval()
    val_rec_loss = val_comp_loss = 0.0
    with torch.no_grad():
        item_upd_v, outfit_upd_v, user_upd_v = model(
            item_embs, item_adj_sparse,
            outfit_embs, outfit_item_adj_sparse,
            user_embs, user_outfit_adj_sparse
        )

        for vb in val_loader:
            u_idx   = vb['user_idx'].to(device)
            o_idx   = vb['outfit_idx'].to(device)
            neg_idx = torch.randint(outfit_upd_v.size(0), o_idx.shape, device=device)
            val_rec_loss += bpr_loss(
                model.score_recommendation(user_upd_v[u_idx], outfit_upd_v[o_idx]),
                model.score_recommendation(user_upd_v[u_idx], outfit_upd_v[neg_idx])
            ).item()
        for cb in compat_val_loader:
            pos_embs = item_upd_v[cb['pos_item_indices'].to(device)]
            neg_embs = item_upd_v[cb['neg_item_indices'].to(device)]
            val_comp_loss += bpr_loss(
                model.compatibility_scorer(pos_embs),
                model.compatibility_scorer(neg_embs)
            ).item()

    # Averages
    tr_rec  = total_rec_loss / n_batches
    tr_comp = total_comp_loss / n_batches
    vl_rec  = val_rec_loss / len(val_loader)
    vl_comp = val_comp_loss / len(compat_val_loader)

    # Metrics (eval every epoch)
# Trong training loop, chỗ gọi evaluate_rec:
    train_m = evaluate_rec(
        model, train_loader,
        item_embs,   item_adj_sparse,       # ← truyền sparse matrix
        outfit_embs, outfit_item_adj_sparse,
        user_embs,   user_outfit_adj_sparse,
        device, k=10
    )
    val_m = evaluate_rec(
        model, val_loader,
        item_embs,   item_adj_sparse,
        outfit_embs, outfit_item_adj_sparse,
        user_embs,   user_outfit_adj_sparse,
        device, k=10
    )
    train_acc = evaluate_compat(model, compat_train_loader, item_upd_v, device)
    val_acc   = evaluate_compat(model, compat_val_loader,   item_upd_v, device)

    # Log
    duration = time.time() - t0
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} [{duration:.0f}s] '
          f'Loss={tr_rec+tr_comp:.4f} | '
          f'Train HR={train_m["HR@K"]:.4f} Prec={train_m["Precision@K"]:.4f} '
          f'Rec={train_m["Recall@K"]:.4f} NDCG={train_m["NDCG@K"]:.4f} Acc={train_acc:.4f} | '
          f'Val HR={val_m["HR@K"]:.4f} Prec={val_m["Precision@K"]:.4f} '
          f'Rec={val_m["Recall@K"]:.4f} NDCG={val_m["NDCG@K"]:.4f} Acc={val_acc:.4f}')

    # Record history
    for key, val in [('train_loss', tr_rec+tr_comp), ('val_loss', vl_rec+vl_comp),
                     ('train_rec_loss', tr_rec), ('val_rec_loss', vl_rec),
                     ('train_comp_loss', tr_comp), ('val_comp_loss', vl_comp),
                     ('train_hr', train_m['HR@K']), ('val_hr', val_m['HR@K']),
                     ('train_ndcg', train_m['NDCG@K']), ('val_ndcg', val_m['NDCG@K']),
                     ('train_prec', train_m['Precision@K']), ('val_prec', val_m['Precision@K']),
                     ('train_rec', train_m['Recall@K']), ('val_rec', val_m['Recall@K']),
                     ('train_auc', train_m['AUC']), ('val_auc', val_m['AUC']),
                     ('train_acc', train_acc), ('val_acc', val_acc)]:
        history[key].append(val)

    scheduler.step(val_m['HR@K'])

    # Save best & early stopping
    if val_m['HR@K'] > best_val_hr:
        best_val_hr = val_m['HR@K']
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  --> Best model saved (Val HR@10={best_val_hr:.4f})')
        early_stop_cnt = 0
    else:
        early_stop_cnt += 1
        if early_stop_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'Training done. Best Val HR@10 = {best_val_hr:.4f}')

Starting training for 100 epochs...


TypeError: H_HFGAT.forward() takes 7 positional arguments but 10 were given

In [ ]:
# ============================================================
# CELL 11: Test evaluation
# ============================================================
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))

with torch.no_grad():
    item_upd_test, outfit_upd_test, user_upd_test = model(
        item_embs, item_adj_sparse,
        outfit_embs, outfit_item_adj_sparse,
        user_embs, user_outfit_adj_sparse
    )

test_m = evaluate_rec(
    model, test_loader,
    item_embs, item_adj_sparse,
    outfit_embs, outfit_item_adj_sparse,
    user_embs, user_outfit_adj_sparse,
    device, k=10
)
test_acc = evaluate_compat(model, compat_test_loader, item_upd_test, device)

print('\n=== TEST RESULTS ===')
print(f'HR@10        : {test_m["HR@K"]:.4f}')
print(f'Precision@10 : {test_m["Precision@K"]:.4f}')
print(f'Recall@10    : {test_m["Recall@K"]:.4f}')
print(f'NDCG@10      : {test_m["NDCG@K"]:.4f}')
print(f'AUC          : {test_m["AUC"]:.4f}')
print(f'Compat Acc   : {test_acc:.4f}')
print('\n=== PAPER TARGETS ===')
print('HR@10=0.4286 | Prec@10=0.4424 | Rec@10=0.1580 | NDCG@10=0.1340 | Acc=0.8956')


In [ ]:
# ============================================================
# CELL 12: Training curves
# ============================================================
sns.set_style('darkgrid')
epochs_ran = list(range(1, len(history['train_loss'])+1))

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('FGAT Training Metrics', fontsize=14)

plots = [
    (axes[0,0], 'Total Loss',         'train_loss',      'val_loss'),
    (axes[0,1], 'Recommendation Loss','train_rec_loss',  'val_rec_loss'),
    (axes[0,2], 'Compatibility Loss', 'train_comp_loss', 'val_comp_loss'),
    (axes[1,0], 'HR@10',              'train_hr',        'val_hr'),
    (axes[1,1], 'NDCG@10',            'train_ndcg',      'val_ndcg'),
    (axes[1,2], 'Precision@10',       'train_prec',      'val_prec'),
    (axes[2,0], 'Recall@10',          'train_rec',       'val_rec'),
    (axes[2,1], 'AUC',                'train_auc',       'val_auc'),
    (axes[2,2], 'Compat Accuracy',    'train_acc',       'val_acc'),
]

for ax, title, train_key, val_key in plots:
    ax.plot(epochs_ran, history[train_key], label='Train')
    ax.plot(epochs_ran, history[val_key],   label='Val', linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to training_curves.png')